# TFG Radiomics NACT — Duke-Breast-Cancer-MRI
## Notebook final, reproducible y congelado

**Pregunta:** predicción **pre-NACT** (antes de iniciar quimioterapia neoadyuvante) de una
**respuesta muy favorable** a la neoadyuvancia en cáncer de mama, combinando variables
clínico-patológicas basales (**C**), hallazgos MRI cualitativos basales (**M**), contexto de
adquisición de la MRI (**A**) y radiómicas extraídas de la MRI pre-tratamiento (**R**).

**Target primario:** `Overall Near-complete Response: Stricter Definition` (0=no respuesta,
1=respuesta completa, 2=respuesta casi completa → ambas cuentan como respondedor).

Este notebook parte **únicamente** de los Excel originales (`datos/Clinical_and_Other_Features.xlsx`,
`datos/Imaging_Features.xlsx`), usa **paths relativos**, y reproduce desde cero el pipeline V2
congelado. No se prueban
variables, modelos, targets, seeds ni thresholds nuevos: todo lo aquí ejecutado ya fue
validado y congelado en Etapas 2–5; este notebook es la versión limpia y unificada de ese
código, no un nuevo experimento. Los análisis opcionales (Etapas 6–7: subgrupo luminal y
sensibilidad al target) quedaron fuera del análisis final y no se incluyen.

**Tratamiento de faltantes (definitivo):** `stage_M` Mx se reconstruye con
*Metastatic at Presentation* (sólo para caracterizar la cohorte: tras la reconstrucción queda
280/281 M0 y se excluye como predictor por varianza casi nula); `histologic_type` y `bilateral_bc` sin dato se codifican como
categoría explícita `NC` (dato no recolectado, no categoría clínica).

Los CSV de `pipeline_v2/stage*_*.csv` se usan **sólo para validar** que este notebook reproduce
los mismos números — nunca como sustituto del cálculo (ver Sección 9 y el bloque de validación
al final).


## 0 · Configuración

Imports, seeds congelados, paths relativos y una configuración gráfica global única,
reutilizada por todas las figuras del notebook (helper `save_figure`).


In [1]:

from __future__ import annotations
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                              precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.optimize import brentq

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

# ---- seeds y config congelados (identicos a Etapas 2-3-4-5-6-7; no elegidos ahora) ----
RANDOM_STATE = 42
BOOT_SEED = 20260909
N_BOOT = 5000
REP_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]   # Etapas 3/4/7 (Stricter/Looser/pCR)
LUMINAL_SEEDS = [42, 43, 44, 45, 46]                    # Etapa 6

# ---- paths relativos (el notebook se ejecuta parado en la raiz del proyecto) ----
ROOT = Path.cwd()
DB_DIR = ROOT / "datos"                    # Excel originales de Duke
CLIN_XLSX = DB_DIR / "Clinical_and_Other_Features.xlsx"
IMG_XLSX = DB_DIR / "Imaging_Features.xlsx"
PIPE_DIR = ROOT / "pipeline_v2"            # solo para VALIDAR contra CSV congelados
FIG_DIR = ROOT / "notebook" / "figuras"
FIG_OPT_DIR = FIG_DIR / "optional"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert CLIN_XLSX.exists() and IMG_XLSX.exists(), "Excel originales no encontrados (paths relativos)"

# ---- estilo grafico global (identidad visual oficial del TFG, aprobada) ----
BG, BG2, TXT = "#FFFFFF", "#F7F9FC", "#263238"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18, "grid.linewidth": 0.6, "grid.color": "#B9C2D0",
    "axes.edgecolor": "#B9C2D0", "axes.labelcolor": TXT,
    "xtick.color": TXT, "ytick.color": TXT, "text.color": TXT,
    "legend.frameon": False, "figure.dpi": 110,
})

# Paleta oficial del TFG (aprobada), identica en TFG_Radiomics_EDA.ipynb y TFG_Radiomics_Final.ipynb
COLOR_C   = "#2A9D8F"   # modelo clinico (C), constante en todo el notebook
COLOR_CR  = "#7B61FF"   # C+R
COLOR_CAR = "#3A86FF"   # C+A+R -- modelo principal, constante en todo el notebook
COLOR_M   = "#5BC0EB"   # M / bloques secundarios (p.ej. bloque A en desgloses)
COLOR_RF  = "#F4C95D"   # Random Forest
COLOR_GB  = "#F26B5B"   # Gradient Boosting
COLOR_REF = "#8A94A6"   # referencias / baselines / treat-all-none
COLOR_POS = "#36C2A6"   # respondedores
COLOR_NEG = "#7895B2"   # no respondedores

MODEL_COLORS = {"C": COLOR_C, "C+R": COLOR_CR, "C+A+R": COLOR_CAR,
                "random_forest": COLOR_RF, "gradient_boosting": COLOR_GB}

FIGURE_CATALOG = []

def save_figure(fig, name, what, analysis, priority, use, optional=False, formats=("png", "pdf")):
    """Guarda una figura en notebook/figuras/ (o /optional), 300dpi, y la registra en el catalogo."""
    d = FIG_OPT_DIR if optional else FIG_DIR
    d.mkdir(exist_ok=True)
    saved = []
    for ext in formats:
        p = d / f"{name}.{ext}"
        fig.savefig(p, dpi=300, bbox_inches="tight")
        saved.append(str(p.relative_to(ROOT)))
    plt.close(fig)
    FIGURE_CATALOG.append({"figura": name, "archivo": saved[0], "que_muestra": what,
                            "analisis": analysis, "prioridad": priority, "uso_potencial": use,
                            "optional": optional})
    print(f"[fig] {name}  ->  {saved}")

print("Config OK.")
print("DB_DIR existe:", DB_DIR.exists(), "| CLIN_XLSX:", CLIN_XLSX.name, "| IMG_XLSX:", IMG_XLSX.name)


Config OK.
DB_DIR existe: True | CLIN_XLSX: Clinical_and_Other_Features.xlsx | IMG_XLSX: Imaging_Features.xlsx


## 1 · Carga de datos

`Clinical_and_Other_Features.xlsx` tiene una fila 0 con los **nombres de columna** y una fila 1
con el **Legend** (codificación de cada variable) — no es un header estándar de una sola fila.
Se lee con `header=None`, se toman los nombres de la fila 0, se descarta la fila 1 (Legend) y el
resto queda como datos. `Imaging_Features.xlsx` no tiene fila de Legend.

Se verifica que `Patient ID` es una clave 1:1 en ambos archivos antes de cualquier merge.


In [2]:

raw = pd.read_excel(CLIN_XLSX, sheet_name="Data", header=None)
# Archivo oficial de TCIA: se quita la fila superior de grupos y se asignan nombres unicos a 16 columnas
# cuyo nombre esta en la fila de grupos o en la leyenda (mismo criterio que pipeline_v2/run_stage2.leer_clinico).
ENCABEZADOS_ARMONIZADOS = {
    23: ("ER", "ER status"), 24: ("PR", "PR status"), 25: ("HER2", "HER2 status"),
    31: ("Tumor Grade", "Tumor Grade (Tubule)"), 32: (None, "Tumor Grade (Nuclear)"),
    33: (None, "Tumor Grade (Mitotic)"),
    38: ("Bilateral Information", "Bilateral breast cancer"), 39: (None, "Bilateral different. Rec status"),
    40: (None, "Bilateral. Side Annotated"),
    41: ("For Other Side If Bilateral", "For Other Side If Bilateral. Side of cancer"),
    42: (None, "Other side oncotype score"), 43: (None, "Other side Norringham grade"),
    44: (None, "Other side ER"), 45: (None, "Other side PR"), 46: (None, "Other side HER2"),
    47: (None, "Other side Mol Subtype"),
}
if raw.iat[0, 0] == "Patient Information":
    raw = raw.iloc[1:].reset_index(drop=True)
    for j, (original, nombre) in ENCABEZADOS_ARMONIZADOS.items():
        actual = raw.iat[0, j]
        assert (pd.isna(actual) if original is None else actual == original), (j, actual)
        raw.iat[0, j] = nombre
col_names = raw.iloc[0].tolist()
clin = raw.iloc[2:].copy()          # fila 1 = Legend -> se descarta, NO son datos
clin.columns = col_names
clin = clin.reset_index(drop=True)

rad = pd.read_excel(IMG_XLSX)       # sin fila de leyenda

ID_COL = "Patient ID"
assert clin[ID_COL].is_unique, "Patient ID no es unico en el archivo clinico"
assert rad[ID_COL].is_unique, "Patient ID no es unico en el archivo de radiomicas"

print(f"Clinico:   {clin.shape[0]} pacientes x {clin.shape[1]} columnas")
print(f"Radiomica: {rad.shape[0]} pacientes x {rad.shape[1]} columnas")
print(f"Interseccion Patient ID: {len(set(clin[ID_COL]) & set(rad[ID_COL]))} "
      f"(merge 1:1 potencial, Patient ID se usa solo para mergear, nunca como predictor)")


Clinico:   922 pacientes x 98 columnas
Radiomica: 922 pacientes x 530 columnas
Interseccion Patient ID: 922 (merge 1:1 potencial, Patient ID se usa solo para mergear, nunca como predictor)


## 2 · Cohorte y target principal

**Momento de predicción:** todas las variables usadas son **pre-tratamiento** (basales, previas
al inicio de la quimioterapia neoadyuvante) — no se usa ninguna variable post-tratamiento,
de seguimiento, ni ninguna variante alternativa del outcome (leakage).

**Filtros de inclusión:** recibió neoadyuvancia (`Received Neoadjuvant Therapy or Not`==1) **y**
quimioterapia neoadyuvante (`Neoadjuvant Chemotherapy`==1) **y** target evaluable
(`Overall Near-complete Response: Stricter Definition` ∈ {0,1,2}; se excluyen 3="pathologic
assessment not available" y NA="no aplica").

**Binarización:** {0}→0 (no respuesta), {1,2}→1 (respuesta completa o casi completa =
"respuesta muy favorable".


In [3]:

TARGET_COL = "Overall Near-complete Response:  Stricter Definition"   # doble espacio real en el Excel

def _to_num(s):
    return pd.to_numeric(s, errors="coerce")

neo = _to_num(clin["Received Neoadjuvant Therapy or Not"])
chemo = _to_num(clin["Neoadjuvant Chemotherapy"])
tgt = _to_num(clin[TARGET_COL])

n_total = len(clin)
n_neo = int((neo == 1).sum())
n_neo_chemo = int(((neo == 1) & (chemo == 1)).sum())
mask = (neo == 1) & (chemo == 1) & (tgt.isin([0, 1, 2]))
n_evaluable = int(mask.sum())
n_excluded_target = int(((neo == 1) & (chemo == 1) & ~tgt.isin([0, 1, 2])).sum())

coh = clin[mask].copy().reset_index(drop=True)
y = (tgt[mask].reset_index(drop=True) != 0).astype(int)
y.name = "y"

print("Flujo de inclusion/exclusion:")
print(f"  Pacientes totales:                         {n_total}")
print(f"  Con neoadyuvancia:                          {n_neo}")
print(f"  Con quimioterapia neoadyuvante:              {n_neo_chemo}")
print(f"  Target Stricter evaluable (0/1/2):           {n_evaluable}")
print(f"  Excluidos (target=3 'no evaluable' / NA):    {n_excluded_target}")


Flujo de inclusion/exclusion:
  Pacientes totales:                         922
  Con neoadyuvancia:                          312
  Con quimioterapia neoadyuvante:              292
  Target Stricter evaluable (0/1/2):           281
  Excluidos (target=3 'no evaluable' / NA):    11


## 3 · Bloques de variables (C / M / A / R) y exclusiones (O / T / X)

- **C** — clínica/patología basal (14 variables: edad, ER/PR/HER2, menopausia, lateralidad,
  bilateralidad, estadios T/N, grados tumorales, raza, tipo histológico).
- **M** — hallazgos MRI cualitativos basales (5 variables: multicentricidad, contralateral,
  adenopatías, piel/pezón, pared torácica). Secundario — ver Sección 5.
- **A** — contexto de adquisición de la MRI (14 variables: TR/TE, field strength, matriz, FOV,
  fabricante/modelo, contraste, etc.). Representa protocolo/equipo, **no** biología tumoral.
- **R** — radiómicas extraídas de la MRI pre-tratamiento (pool de 529 crudas).
- **O** (mamografía/ecografía), **T** (tratamiento/post-tratamiento) y **X** (variables dudosas
  o de alta proporción de missing) — **excluidos** del modelo por representar información no
  disponible pre-tratamiento, redundante, o de calidad insuficiente (detalle completo en
  `AUDITORIA_PIPELINE_BASE.md`).

`Patient ID` se usa **exclusivamente** para el merge 1:1 clínico↔radiómicas y luego se excluye
como predictor.

**Radiómicas:** 529 crudas → **dedup exacto** (columnas idénticas byte-a-byte, sin usar el
target) → 509 en el pool. Dentro de cada fold de CV se aplica además un filtro de correlación
(|r|>0.95, ajustado **sólo con el train del fold**) que deja habitualmente ~298 variables, y
luego PCA (también ajustado sólo con el train del fold); el número de componentes lo elige el
grid search en cada fold (en el modelo principal C+A+R, en la mayoría de los folds entre 1 y 3). El dedup es **global y
target-blind** (no depende del split); el filtro de correlación y el PCA son **fold/train-only**.


In [4]:

# ---- C: derivadas / limpias ----
C_CONTINUOUS = {"age": None}                                # derivada de 'Date of Birth (Days)'
C_BINARY = {
    "ER_bin": "ER status", "PR_bin": "PR status", "HER2_bin": "HER2 status",
    "menopause_post": "Menopause (at diagnosis)",            # 0=pre 1=post ; 2(N/A)->missing
    "tumor_side_R": "Tumor Location",                        # L/R ; reconstruida desde 'Position'
}
C_ORDINAL = {
    "stage_T": "Staging(Tumor Size)# [T]",
    "stage_N": "Staging(Nodes)#(Nx replaced by -1)[N]",       # -1 (Nx) -> missing
    # stage_M: se reconstruye (Mx -> 'Metastatic at Presentation') pero NO es predictor: practicamente
    # constante (280/281 M0, 1 M1) -> excluida por varianza casi nula.
    "grade_tubule": "Tumor Grade (Tubule)",
    "grade_nuclear": "Tumor Grade (Nuclear)",
    "grade_mitotic": "Tumor Grade (Mitotic)",
}
C_NOMINAL = {"race": "Race and Ethnicity",
             "histologic_type": "Histologic type",        # vacio -> NC_CODE
             "bilateral_bc": "Bilateral breast cancer"}   # 0/1 ; NC -> NC_CODE (nominal de 3 niveles)
# "NC" = Data Not Collected (leyenda de Duke): dato no recolectado, NO categoria clinica ni ausencia.
NC_CODE = -99

M_BINARY = {
    "mf_multicentric": "Multicentric/Multifocal",
    "mf_contralateral": "Contralateral Breast Involvement",
    "mf_lymphadenopathy": "Lymphadenopathy or Suspicious Nodes",
    "mf_skin_nipple": "Skin/Nipple Invovlement",
    "mf_pec_chest": "Pec/Chest Involvement",
}

A_CONTINUOUS = {
    "days_to_mri": "Days to MRI (From the Date of Diagnosis)",
    "tr": "TR (Repetition Time)", "te": "TE (Echo Time)",
}
A_ORDINAL = {
    "field_strength": "Field Strength (Tesla)", "slice_thickness": "Slice Thickness ",
    "matrix_rows": "Rows", "flip_angle": "Flip Angle \n",
    "fov": "FOV Computed (Field of View) in cm ",
}
A_BINARY = {"patient_position": "Patient Position During MRI"}
A_NOMINAL = {
    "manufacturer": "Manufacturer", "model_name": "Manufacturer Model Name",
    "scan_options": "Scan Options", "acq_matrix": "Acquisition Matrix",
    "contrast_agent": "Contrast Agent",
}

BLOCK_TYPES = {
    "C": {"continuous": list(C_CONTINUOUS), "binary": list(C_BINARY),
          "ordinal": list(C_ORDINAL), "nominal": list(C_NOMINAL)},
    "M": {"binary": list(M_BINARY)},
    "A": {"continuous": list(A_CONTINUOUS), "binary": list(A_BINARY),
          "ordinal": list(A_ORDINAL), "nominal": list(A_NOMINAL)},
}

out = pd.DataFrame({ID_COL: coh[ID_COL].values})

# ---- C ----
out["age"] = -_to_num(coh["Date of Birth (Days)"]) / 365.25
men = _to_num(coh["Menopause (at diagnosis)"]); men = men.where(men != 2, np.nan)
out["menopause_post"] = men
race = _to_num(coh["Race and Ethnicity"]); race = race.where(race != 0, np.nan)
out["race"] = race
for dst, src in {"ER_bin": "ER status", "PR_bin": "PR status", "HER2_bin": "HER2 status"}.items():
    out[dst] = _to_num(coh[src])
# histologic_type / bilateral_bc: faltantes = bloque "no recolectado" de Duke (NC), sin variable
# equivalente para reconstruirlos -> categoria explicita NC en vez de imputar con la moda.
bil_raw = coh["Bilateral breast cancer"]
out["bilateral_bc"] = _to_num(bil_raw).where(bil_raw.astype(str).str.strip() != "NC", NC_CODE)
out["histologic_type"] = _to_num(coh["Histologic type"]).fillna(NC_CODE)

tl = coh["Tumor Location"].astype(str).str.strip().str.upper()
tl = tl.where(tl.isin(["L", "R"]), np.nan)
pos_side = coh["Position"].astype(str).str.strip().str.upper().str.extract(r"^\s*([LR])")[0]
side = tl.fillna(pos_side)
out["tumor_side_R"] = side.map({"L": 0.0, "R": 1.0})

out["stage_T"] = _to_num(coh["Staging(Tumor Size)# [T]"])
n_ = _to_num(coh["Staging(Nodes)#(Nx replaced by -1)[N]"]); out["stage_N"] = n_.where(n_ >= 0, np.nan)
# stage_M: Mx (-1) se recupera con 'Metastatic at Presentation' (misma fuente clinica que el TNM,
# pretratamiento; concordancia 718/719 con M observado en Duke completo): 0 -> M0, 1 -> M1.
m_ = _to_num(coh["Staging(Metastasis)#(Mx -replaced by -1)[M]"])
out["stage_M"] = m_.where(m_ >= 0, _to_num(coh["Metastatic at Presentation (Outside of Lymph Nodes)"]))
for dst, src in {"grade_tubule": "Tumor Grade (Tubule)", "grade_nuclear": "Tumor Grade (Nuclear)",
                 "grade_mitotic": "Tumor Grade (Mitotic)"}.items():
    out[dst] = _to_num(coh[src])

# ---- M ----
for dst, src in M_BINARY.items():
    out[dst] = _to_num(coh[src])

# ---- A ----
for dst, src in {**A_CONTINUOUS, **A_ORDINAL, **A_BINARY, **A_NOMINAL}.items():
    out[dst] = _to_num(coh[src])

# ---- R: dedup exacto (global, target-blind) ----
def dedup_radiomics(rad_df):
    feats = [c for c in rad_df.columns if c != ID_COL]
    seen, keep, dropped = {}, [], []
    for c in feats:
        key = tuple(pd.util.hash_pandas_object(rad_df[c].fillna(-9.99e12), index=False))
        if key in seen:
            dropped.append(c)
        else:
            seen[key] = c
            keep.append(c)
    return rad_df[[ID_COL] + keep].copy(), dropped

rad_dedup, dropped_names = dedup_radiomics(rad)
rad_cols = [c for c in rad_dedup.columns if c != ID_COL]

# 'y' se adjunta como columna a 'out' ANTES del merge para garantizar alineacion correcta
# con Patient ID, sin depender de que el merge preserve el orden de filas.
out["_y"] = y.values
merged = out.merge(rad_dedup, on=ID_COL, how="inner")
y = merged.pop("_y").astype(int)

n_final = len(merged)
print(f"Radiomicas: {len(rad.columns)-1} crudas -> dedup exacto (-{len(dropped_names)}) -> "
      f"{len(rad_cols)} en el pool")
print(f"Cohorte final tras merge con radiomicas: N={n_final}")
print(f"  Respondedores (y=1): {int(y.sum())}   No respondedores (y=0): {int((y==0).sum())}   "
      f"Prevalencia: {y.mean()*100:.1f}%")

assert n_final == 281, f"cohorte reconstruida = {n_final}, se esperaba 281"
assert int(y.sum()) == 93 and int((y == 0).sum()) == 188, "distribucion de clases no coincide con el congelado (93/188)"
print("OK: N=281, 93 respondedores, 188 no respondedores (coincide con el congelado en Etapas 2-6).")


Radiomicas: 529 crudas -> dedup exacto (-20) -> 509 en el pool
Cohorte final tras merge con radiomicas: N=281
  Respondedores (y=1): 93   No respondedores (y=0): 188   Prevalencia: 33.1%
OK: N=281, 93 respondedores, 188 no respondedores (coincide con el congelado en Etapas 2-6).


## 4 · Preprocessing (Pipeline V2 congelado)

- **Continuas/ordinales:** imputación por mediana + escalado estándar.
- **Binarias:** imputación por moda (sin escalar).
- **Nominales:** imputación por moda + one-hot encoding. En `histologic_type` y `bilateral_bc` los faltantes ya llegan como categoría explícita `NC` (dato no recolectado), por lo que no se imputan.
- **stage_M:** Mx se recupera con `Metastatic at Presentation`, pero no se usa como predictor (280/281 M0, 1 M1: varianza casi nula).
- **Radiómicas (R):** imputación por mediana → **filtro de correlación** (elimina varianza cero
  y una de cada par con |Pearson r|>0.95) → escalado → **PCA**. El filtro de correlación y el
  PCA se ajustan **exclusivamente sobre el train de cada fold** (`CorrelationFilter` es un
  `TransformerMixin` de scikit-learn, se incluye dentro del `Pipeline`/`ColumnTransformer`).
- **Modelo:** Logistic Regression L2, `class_weight="balanced"`.
- **Grillas:** `C ∈ {0.01, 0.1, 1, 10}`; si el bloque R está presente,
  `n_components ∈ {1, 2, 3, 5, 10, 15, 20, 30}`. La grilla de PCA original era `{5, 10, 15, 20, 30}`;
  se amplió hacia abajo porque la selección quedaba casi siempre en su mínimo (5), es decir, en el
  límite de la grilla.


In [5]:

CORR_THRESHOLD = 0.95
C_GRID = [0.01, 0.1, 1.0, 10.0]
NCOMP_GRID = [1, 2, 3, 5, 10, 15, 20, 30]

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """Elimina varianza cero y una de cada par con |Pearson r| > threshold.
    Se ajusta SOLO con el train que recibe en fit (dentro del pipeline/fold)."""
    def __init__(self, threshold: float = 0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        p = X.shape[1]
        std = X.std(axis=0)
        drop = set(np.where(std == 0)[0].tolist())
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.abs(np.corrcoef(X, rowvar=False))
        corr = np.nan_to_num(corr)
        for j in range(p):
            if j in drop:
                continue
            for i in range(j):
                if i in drop:
                    continue
                if corr[i, j] > self.threshold:
                    drop.add(j)
                    break
        self.keep_ = np.array([k for k in range(p) if k not in drop], dtype=int)
        self.n_in_, self.n_out_ = p, len(self.keep_)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.keep_]


def make_preprocessor(blocks, rad_cols):
    cont, binc, ordc, nomc = [], [], [], []
    for b in ("C", "M", "A"):
        if b in blocks:
            t = BLOCK_TYPES[b]
            cont += t.get("continuous", []); binc += t.get("binary", [])
            ordc += t.get("ordinal", []); nomc += t.get("nominal", [])
    tr = []
    if cont:
        tr.append(("cont", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), cont))
    if binc:
        tr.append(("bin", SimpleImputer(strategy="most_frequent"), binc))
    if ordc:
        tr.append(("ord", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ordc))
    if nomc:
        tr.append(("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nomc))
    if "R" in blocks:
        tr.append(("R", Pipeline([("imp", SimpleImputer(strategy="median")),
                                  ("corr", CorrelationFilter(CORR_THRESHOLD)),
                                  ("sc", StandardScaler()),
                                  ("pca", PCA(random_state=RANDOM_STATE))]), rad_cols))
    return ColumnTransformer(tr, remainder="drop")


def make_estimator(blocks, rad_cols):
    return Pipeline([("preprocessor", make_preprocessor(blocks, rad_cols)),
                     ("clf", LogisticRegression(penalty="l2", class_weight="balanced",
                                                max_iter=5000, random_state=RANDOM_STATE))])


def make_grid(blocks):
    grid = {"clf__C": C_GRID}
    if "R" in blocks:
        grid["preprocessor__R__pca__n_components"] = NCOMP_GRID
    return grid

print("Preprocessing V2 definido (CorrelationFilter, make_preprocessor, make_estimator, make_grid).")


Preprocessing V2 definido (CorrelationFilter, make_preprocessor, make_estimator, make_grid).


## 5 · Análisis principal — hold-out 80/20 (comparabilidad histórica)

Split estratificado 80/20 (`random_state=42`, idéntico a Etapas 2–5). Para cada configuración
de bloques se ajusta un `GridSearchCV` (scoring ROC-AUC, inner CV de 5 folds) **sobre el train**
y se evalúa una sola vez sobre el test. Este esquema se reporta **sólo por comparabilidad con
el histórico del proyecto** — la evaluación principal es la Repeated Nested CV de la Sección 6.

Se incluye también, como **documentación secundaria** (no como candidato a modelo principal),
C+M: en Etapa 2 se observó que M no aporta valor incremental relevante sobre C.


In [6]:

MODELS_MAIN = {"C": ["C"], "C+R": ["C", "R"], "C+A+R": ["C", "A", "R"]}
MODELS_SECONDARY = {"C+M": ["C", "M"]}   # solo documentacion

idx = np.arange(len(merged))
tr_idx, te_idx = train_test_split(idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

holdout_rows = []
holdout_fits = {}
for name, blocks in {**MODELS_MAIN, **MODELS_SECONDARY}.items():
    gs = GridSearchCV(make_estimator(blocks, rad_cols), make_grid(blocks),
                      scoring="roc_auc", cv=inner_cv, n_jobs=-1, refit=True)
    gs.fit(merged.iloc[tr_idx], y.iloc[tr_idx])
    proba_te = gs.predict_proba(merged.iloc[te_idx])[:, 1]
    holdout_fits[name] = (gs.best_estimator_, proba_te)
    holdout_rows.append({
        "model": name, "roc_auc": roc_auc_score(y.iloc[te_idx], proba_te),
        "pr_auc": average_precision_score(y.iloc[te_idx], proba_te),
        "best_C": gs.best_params_["clf__C"],
        "best_n_components": gs.best_params_.get("preprocessor__R__pca__n_components", None),
    })
    print(f"  [holdout] {name:6s} ROC={holdout_rows[-1]['roc_auc']:.3f}  "
          f"PR={holdout_rows[-1]['pr_auc']:.3f}  C={holdout_rows[-1]['best_C']}  "
          f"nPC={holdout_rows[-1]['best_n_components']}")

holdout_df = pd.DataFrame(holdout_rows).set_index("model")
display(holdout_df)


  [holdout] C      ROC=0.778  PR=0.634  C=0.1  nPC=None


  [holdout] C+R    ROC=0.745  PR=0.556  C=1.0  nPC=5


  [holdout] C+A+R  ROC=0.784  PR=0.708  C=0.1  nPC=2
  [holdout] C+M    ROC=0.771  PR=0.608  C=0.1  nPC=None


,roc_auc,pr_auc,best_C,best_n_components
model,,,,
C,0.778393,0.633772,0.1,NaN
C+R,0.745152,0.555596,1.0,5.0
C+A+R,0.783934,0.708269,0.1,2.0
C+M,0.771468,0.608132,0.1,NaN


## 6 · Validación — Repeated Nested CV (evaluación principal)

**Ésta es la estimación principal del desempeño**, no el hold-out. Esquema: outer
`StratifiedKFold(5)` × inner `StratifiedKFold(5)`, **10 repeticiones** con seeds predefinidos
`42..51` (nunca elegidos por resultado). Todo el preprocessing se ajusta dentro del train de
cada fold. Se agregan las probabilidades OOF promediando entre repeticiones (**OOF agregado**),
y se calcula un bootstrap pareado a nivel paciente (5000 resamples, seed `20260909`) para el
Δ ROC-AUC / Δ PR-AUC de C+R y C+A+R frente a C.

**No se mezclan** en una misma tabla el hold-out, la media±SD por-repetición y el OOF agregado:
cada uno se reporta por separado y rotulado.


In [7]:

N_OUTER, N_INNER = 5, 5

def fit_predict_fold(blocks, Xtr, ytr, Xte, inner_seed):
    gs = GridSearchCV(make_estimator(blocks, rad_cols), make_grid(blocks), scoring="roc_auc",
                      cv=StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=inner_seed),
                      n_jobs=-1, refit=True)
    gs.fit(Xtr, ytr)
    proba = gs.predict_proba(Xte)[:, 1]
    best = gs.best_estimator_
    n_rad = None
    if "R" in blocks:
        n_rad = int(best.named_steps["preprocessor"].named_transformers_["R"].named_steps["corr"].n_out_)
    chosen = {"C": gs.best_params_["clf__C"],
              "n_components": gs.best_params_.get("preprocessor__R__pca__n_components", None),
              "n_radiomics_after_corr": n_rad}
    return proba, chosen


def repeated_nested_cv(models, merged_df, y_ser, seeds, n_outer, n_inner):
    yv = y_ser.values
    n = len(merged_df)
    per_rep, stability = [], []
    oof_stack = {m: np.full((len(seeds), n), np.nan) for m in models}
    for ri, seed in enumerate(seeds):
        outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=seed)
        oof_rep = {m: np.full(n, np.nan) for m in models}
        for fi, (tri, tei) in enumerate(outer.split(merged_df, yv)):
            Xtr, Xte = merged_df.iloc[tri], merged_df.iloc[tei]
            ytr = y_ser.iloc[tri]
            for m, blocks in models.items():
                proba, chosen = fit_predict_fold(blocks, Xtr, ytr, Xte, inner_seed=seed)
                oof_rep[m][tei] = proba
                stability.append({"model": m, "repetition": ri, "seed": seed, "outer_fold": fi, **chosen})
        for m in models:
            oof_stack[m][ri] = oof_rep[m]
            per_rep.append({"model": m, "repetition": ri, "seed": seed,
                            "roc_auc": roc_auc_score(yv, oof_rep[m]),
                            "pr_auc": average_precision_score(yv, oof_rep[m])})
        print(f"    rep {ri + 1}/{len(seeds)} (seed {seed}) lista")
    agg = pd.DataFrame({"patient_id": merged_df[ID_COL].values, "y_true": yv})
    for m in models:
        agg[m] = oof_stack[m].mean(axis=0)
    return pd.DataFrame(per_rep), pd.DataFrame(stability), agg


def summarize_per_rep(per_rep, models, ref="C"):
    rows = []
    piv_roc = per_rep.pivot(index="repetition", columns="model", values="roc_auc")
    piv_pr = per_rep.pivot(index="repetition", columns="model", values="pr_auc")
    for m in models:
        r, p = piv_roc[m], piv_pr[m]
        rows.append({"model": m, "roc_auc_mean": r.mean(), "roc_auc_sd": r.std(ddof=1),
                     "roc_auc_min": r.min(), "roc_auc_max": r.max(),
                     "pr_auc_mean": p.mean(), "pr_auc_sd": p.std(ddof=1),
                     "pr_auc_min": p.min(), "pr_auc_max": p.max()})
    return pd.DataFrame(rows).set_index("model")


def aggregated_metrics(agg, models):
    y_ = agg["y_true"].values
    return {m: {"roc_auc_agg": roc_auc_score(y_, agg[m].values),
                "pr_auc_agg": average_precision_score(y_, agg[m].values)} for m in models}


def paired_bootstrap(agg, ref, others, n_boot=N_BOOT, seed=BOOT_SEED):
    y_ = agg["y_true"].values
    n = len(y_)
    rng = np.random.default_rng(seed)
    probs = {m: agg[m].values for m in [ref] + others}
    roc0 = {m: roc_auc_score(y_, probs[m]) for m in probs}
    pr0 = {m: average_precision_score(y_, probs[m]) for m in probs}
    rows = []
    for m in others:
        d_roc, d_pr = [], []
        b = 0
        rng_m = np.random.default_rng(seed)
        while b < n_boot:
            idx_ = rng_m.integers(0, n, n)
            yb = y_[idx_]
            if yb.min() == yb.max():
                continue
            rc = roc_auc_score(yb, probs[ref][idx_]); pc = average_precision_score(yb, probs[ref][idx_])
            d_roc.append(roc_auc_score(yb, probs[m][idx_]) - rc)
            d_pr.append(average_precision_score(yb, probs[m][idx_]) - pc)
            b += 1
        d_roc, d_pr = np.array(d_roc), np.array(d_pr)
        rows.append({"comparison": f"{ref} vs {m}",
                     "delta_roc_point": roc0[m] - roc0[ref],
                     "delta_roc_ci_lo": np.percentile(d_roc, 2.5), "delta_roc_ci_hi": np.percentile(d_roc, 97.5),
                     "delta_pr_point": pr0[m] - pr0[ref],
                     "delta_pr_ci_lo": np.percentile(d_pr, 2.5), "delta_pr_ci_hi": np.percentile(d_pr, 97.5),
                     "n_boot": n_boot})
    return pd.DataFrame(rows)


print("Repeated Nested CV -- Stricter (target principal), modelos C / C+R / C+A+R ...")
per_rep, stability, agg = repeated_nested_cv(MODELS_MAIN, merged, y, REP_SEEDS, N_OUTER, N_INNER)
summ = summarize_per_rep(per_rep, MODELS_MAIN)
agg_m = aggregated_metrics(agg, MODELS_MAIN)
boot = paired_bootstrap(agg, "C", ["C+R", "C+A+R"])

print("\n--- Media +- SD por repeticion (10 reps) ---")
display(summ)
print("\n--- OOF agregado ---")
display(pd.DataFrame(agg_m).T)
print("\n--- Bootstrap pareado (Delta + IC95%) ---")
display(boot)


Repeated Nested CV -- Stricter (target principal), modelos C / C+R / C+A+R ...


    rep 1/10 (seed 42) lista


    rep 2/10 (seed 43) lista


    rep 3/10 (seed 44) lista


    rep 4/10 (seed 45) lista


...

,roc_auc_mean,roc_auc_sd,roc_auc_min,roc_auc_max,pr_auc_mean,pr_auc_sd,pr_auc_min,pr_auc_max
model,,,,,,,,
C,0.741261,0.011612,0.714882,0.757149,0.541588,0.016613,0.503946,0.560305
C+R,0.739333,0.016264,0.714367,0.767445,0.536933,0.025192,0.490567,0.562756
C+A+R,0.742896,0.018570,0.710364,0.764699,0.555483,0.027062,0.502683,0.592581



--- OOF agregado ---


,roc_auc_agg,pr_auc_agg
C,0.749886,0.546409
C+R,0.748856,0.549439
C+A+R,0.761039,0.576586



--- Bootstrap pareado (Delta + IC95%) ---


,comparison,delta_roc_point,delta_roc_ci_lo,delta_roc_ci_hi,delta_pr_point,delta_pr_ci_lo,delta_pr_ci_hi,n_boot
0,C vs C+R,-0.001030,-0.012719,0.010902,0.003030,-0.018216,0.023361,5000
1,C vs C+A+R,0.011153,-0.024411,0.048385,0.030176,-0.032654,0.093796,5000


Figuras de desempeño (evaluación principal, OOF agregado) y de estabilidad entre repeticiones.

In [8]:

# fig 6: ROC OOF C vs C+A+R
fig, ax = plt.subplots(figsize=(5.2, 4.6))
for lab, col in [("C", COLOR_C), ("C+A+R", COLOR_CAR)]:
    fpr, tpr, _ = roc_curve(agg.y_true, agg[lab])
    ax.plot(fpr, tpr, color=col, lw=2.2, label=f"{lab} (AUC={agg_m[lab]['roc_auc_agg']:.3f})")
ax.plot([0, 1], [0, 1], "--", color=COLOR_REF, lw=1)
ax.set_xlabel("1 - especificidad"); ax.set_ylabel("sensibilidad"); ax.legend(loc="lower right")
save_figure(fig, "06_roc_oof_C_vs_CAR", "Curva ROC sobre OOF agregado, C vs C+A+R (Stricter)",
            "Etapa 3 (Repeated Nested CV)", "PRINCIPAL", "Figura principal de desempeño del TFG")

# fig 7: PR OOF C vs C+A+R + prevalencia
fig, ax = plt.subplots(figsize=(5.2, 4.6))
for lab, col in [("C", COLOR_C), ("C+A+R", COLOR_CAR)]:
    prec, rec, _ = precision_recall_curve(agg.y_true, agg[lab])
    ax.plot(rec, prec, color=col, lw=2.2, label=f"{lab} (AUC={agg_m[lab]['pr_auc_agg']:.3f})")
ax.axhline(agg.y_true.mean(), ls="--", color=COLOR_REF, lw=1, label=f"prevalencia ({agg.y_true.mean():.2f})")
ax.set_xlabel("sensibilidad (recall)"); ax.set_ylabel("precision (PPV)"); ax.legend(loc="upper right")
save_figure(fig, "07_pr_oof_C_vs_CAR", "Curva PR sobre OOF agregado, C vs C+A+R + baseline de prevalencia",
            "Etapa 3 (Repeated Nested CV)", "PRINCIPAL", "Figura principal de desempeño del TFG")

# fig 8/9: ROC y PR de los tres modelos (C, C+R, C+A+R)
fig, ax = plt.subplots(figsize=(5.4, 4.7))
for lab, col in [("C", COLOR_C), ("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    fpr, tpr, _ = roc_curve(agg.y_true, agg[lab])
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{lab} (AUC={agg_m[lab]['roc_auc_agg']:.3f})")
ax.plot([0, 1], [0, 1], "--", color=COLOR_REF, lw=1)
ax.set_xlabel("1 - especificidad"); ax.set_ylabel("sensibilidad"); ax.legend(loc="lower right")
save_figure(fig, "08_roc_C_CR_CAR", "Curva ROC OOF agregado: C, C+R, C+A+R",
            "Etapa 3", "PRINCIPAL", "Muestra la incrementalidad R y A+R")

fig, ax = plt.subplots(figsize=(5.4, 4.7))
for lab, col in [("C", COLOR_C), ("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    prec, rec, _ = precision_recall_curve(agg.y_true, agg[lab])
    ax.plot(rec, prec, color=col, lw=2, label=f"{lab} (AUC={agg_m[lab]['pr_auc_agg']:.3f})")
ax.axhline(agg.y_true.mean(), ls="--", color=COLOR_REF, lw=1)
ax.set_xlabel("sensibilidad (recall)"); ax.set_ylabel("precision (PPV)"); ax.legend(loc="upper right")
save_figure(fig, "09_pr_C_CR_CAR", "Curva PR OOF agregado: C, C+R, C+A+R",
            "Etapa 3", "PRINCIPAL", "Muestra la incrementalidad R y A+R")

# fig 10: holdout vs validacion anidada repetida (mismo modelo, dos esquemas, sin mezclar en una sola cifra)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
xs = np.arange(3); labels = ["C", "C+R", "C+A+R"]
w = 0.35
ax.bar(xs - w/2, [holdout_df.loc[m, "roc_auc"] for m in labels], width=w, color=COLOR_REF, label="Hold-out 80/20 (comparabilidad)")
ax.bar(xs + w/2, [agg_m[m]["roc_auc_agg"] for m in labels], width=w, color=COLOR_CAR, label="OOF agregado (validación anidada repetida 10×5×5)")
ax.set_xticks(xs); ax.set_xticklabels(labels); ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.5, 0.85)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1, fontsize=9)
save_figure(fig, "10_holdout_vs_robusto", "ROC-AUC: hold-out histórico vs OOF agregado (Repeated Nested CV)",
            "Etapas 2 y 3", "PRINCIPAL", "Aclara que el hold-out es solo comparabilidad, no el resultado principal")

# fig 11: Delta ROC/PR con IC95% (bootstrap) -- forest/errorbar
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, metric, title in [(axes[0], "roc", "ΔROC-AUC vs C"), (axes[1], "pr", "ΔPR-AUC vs C")]:
    ys = np.arange(len(boot))
    point = boot[f"delta_{metric}_point"].values
    lo = boot[f"delta_{metric}_ci_lo"].values; hi = boot[f"delta_{metric}_ci_hi"].values
    colors = [COLOR_CR if "C+R" in c else COLOR_CAR for c in boot["comparison"]]
    for yi, pi, li, hi_, col in zip(ys, point, lo, hi, colors):
        ax.errorbar([pi], [yi], xerr=[[pi - li], [hi_ - pi]], fmt="o", color=col, ecolor=col, capsize=4)
    ax.axvline(0, color=COLOR_REF, lw=1, ls="--")
    ax.set_yticks(ys); ax.set_yticklabels([c.replace("C vs ", "") for c in boot["comparison"]])
    ax.set_title(title); ax.set_xlabel("Δ (bootstrap, IC95%)")
fig.tight_layout()
save_figure(fig, "11_delta_bootstrap_forest", "Δ ROC/PR-AUC (C+R y C+A+R vs C) con IC95% bootstrap",
            "Etapa 3", "PRINCIPAL", "Resume incertidumbre de la incrementalidad")

# fig 12/13: ROC-AUC y PR-AUC por repeticion
fig, ax = plt.subplots(figsize=(6, 4))
for lab, col in [("C", COLOR_C), ("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    vals = per_rep[per_rep.model == lab]["roc_auc"].values
    ax.scatter(np.full(len(vals), lab), vals, color=col, alpha=0.75, s=40)
    ax.hlines(vals.mean(), -0.2 + list(MODELS_MAIN).index(lab), 0.2 + list(MODELS_MAIN).index(lab), color="black")
ax.set_ylabel("ROC-AUC por repetición"); ax.set_xlabel("")
save_figure(fig, "12_roc_por_repeticion", "ROC-AUC por repetición (10) para C/C+R/C+A+R",
            "Etapa 3", "SECUNDARIA", "Muestra dispersión entre repeticiones")

fig, ax = plt.subplots(figsize=(6, 4))
for lab, col in [("C", COLOR_C), ("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    vals = per_rep[per_rep.model == lab]["pr_auc"].values
    ax.scatter(np.full(len(vals), lab), vals, color=col, alpha=0.75, s=40)
    ax.hlines(vals.mean(), -0.2 + list(MODELS_MAIN).index(lab), 0.2 + list(MODELS_MAIN).index(lab), color="black")
ax.set_ylabel("PR-AUC por repetición")
save_figure(fig, "13_pr_por_repeticion", "PR-AUC por repetición (10) para C/C+R/C+A+R",
            "Etapa 3", "SECUNDARIA", "Muestra dispersión entre repeticiones")

# fig 14/15: Delta por repeticion vs C
piv_roc = per_rep.pivot(index="repetition", columns="model", values="roc_auc")
piv_pr = per_rep.pivot(index="repetition", columns="model", values="pr_auc")
fig, ax = plt.subplots(figsize=(6, 4))
for lab, col in [("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    d = piv_roc[lab] - piv_roc["C"]
    ax.plot(d.index, d.values, "o-", color=col, label=lab)
ax.axhline(0, color=COLOR_REF, lw=1, ls="--")
ax.set_xlabel("repetición"); ax.set_ylabel("Δ ROC-AUC vs C"); ax.legend()
save_figure(fig, "14_delta_roc_por_repeticion", "Δ ROC-AUC vs C, por repetición",
            "Etapa 3", "SECUNDARIA", "Estabilidad de la incrementalidad entre repeticiones")

fig, ax = plt.subplots(figsize=(6, 4))
for lab, col in [("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    d = piv_pr[lab] - piv_pr["C"]
    ax.plot(d.index, d.values, "o-", color=col, label=lab)
ax.axhline(0, color=COLOR_REF, lw=1, ls="--")
ax.set_xlabel("repetición"); ax.set_ylabel("Δ PR-AUC vs C"); ax.legend()
save_figure(fig, "15_delta_pr_por_repeticion", "Δ PR-AUC vs C, por repetición",
            "Etapa 3", "SECUNDARIA", "Estabilidad de la incrementalidad entre repeticiones")

# fig 17: estabilidad n_components PCA
fig, ax = plt.subplots(figsize=(6, 4))
for lab, col in [("C+R", COLOR_CR), ("C+A+R", COLOR_CAR)]:
    vc = stability[stability.model == lab]["n_components"].value_counts().sort_index()
    ax.bar(vc.index.astype(str) + f" ({lab})", vc.values, color=col, alpha=0.8, label=lab)
ax.set_ylabel("nº de folds (de 50)"); ax.set_xlabel("n_components PCA elegido"); ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
save_figure(fig, "17_estabilidad_pca_ncomp", "Frecuencia del n_components de PCA elegido por fold",
            "Etapa 3", "SUPLEMENTARIA", "Chequeo de estabilidad del hiperparámetro")

# fig 18: radiomicas retenidas por el filtro de correlacion
fig, ax = plt.subplots(figsize=(6, 4))
rr = stability[stability.model == "C+A+R"]["n_radiomics_after_corr"].dropna()
ax.hist(rr, bins=15, color=COLOR_CAR, alpha=0.85)
ax.set_xlabel("nº de radiómicas retenidas tras filtro de correlación"); ax.set_ylabel("nº de folds")
save_figure(fig, "18_radiomicas_retenidas_filtro", "Distribución de radiómicas retenidas tras el filtro de correlación (C+A+R)",
            "Etapa 3", "SUPLEMENTARIA", "Evidencia de estabilidad del filtro de correlación")

# fig 19: estabilidad de C (logistic)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, lab, col in zip(axes, ["C", "C+R", "C+A+R"], [COLOR_C, COLOR_CR, COLOR_CAR]):
    vc = stability[stability.model == lab]["C"].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color=col)
    ax.set_title(lab); ax.set_xlabel("C elegido")
axes[0].set_ylabel("nº de folds (de 50)")
fig.tight_layout()
save_figure(fig, "19_estabilidad_C_logistic", "Frecuencia del hiperparámetro C elegido por fold, por modelo",
            "Etapa 3", "SUPLEMENTARIA", "Chequeo de estabilidad del hiperparámetro")

print("Figuras de la Seccion 6 generadas.")


[fig] 06_roc_oof_C_vs_CAR  ->  ['notebook/figuras/06_roc_oof_C_vs_CAR.png', 'notebook/figuras/06_roc_oof_C_vs_CAR.pdf']
[fig] 07_pr_oof_C_vs_CAR  ->  ['notebook/figuras/07_pr_oof_C_vs_CAR.png', 'notebook/figuras/07_pr_oof_C_vs_CAR.pdf']


[fig] 08_roc_C_CR_CAR  ->  ['notebook/figuras/08_roc_C_CR_CAR.png', 'notebook/figuras/08_roc_C_CR_CAR.pdf']
[fig] 09_pr_C_CR_CAR  ->  ['notebook/figuras/09_pr_C_CR_CAR.png', 'notebook/figuras/09_pr_C_CR_CAR.pdf']


[fig] 10_holdout_vs_robusto  ->  ['notebook/figuras/10_holdout_vs_robusto.png', 'notebook/figuras/10_holdout_vs_robusto.pdf']
[fig] 11_delta_bootstrap_forest  ->  ['notebook/figuras/11_delta_bootstrap_forest.png', 'notebook/figuras/11_delta_bootstrap_forest.pdf']


[fig] 12_roc_por_repeticion  ->  ['notebook/figuras/12_roc_por_repeticion.png', 'notebook/figuras/12_roc_por_repeticion.pdf']
[fig] 13_pr_por_repeticion  ->  ['notebook/figuras/13_pr_por_repeticion.png', 'notebook/figuras/13_pr_por_repeticion.pdf']


[fig] 14_delta_roc_por_repeticion  ->  ['notebook/figuras/14_delta_roc_por_repeticion.png', 'notebook/figuras/14_delta_roc_por_repeticion.pdf']
[fig] 15_delta_pr_por_repeticion  ->  ['notebook/figuras/15_delta_pr_por_repeticion.png', 'notebook/figuras/15_delta_pr_por_repeticion.pdf']


[fig] 17_estabilidad_pca_ncomp  ->  ['notebook/figuras/17_estabilidad_pca_ncomp.png', 'notebook/figuras/17_estabilidad_pca_ncomp.pdf']
[fig] 18_radiomicas_retenidas_filtro  ->  ['notebook/figuras/18_radiomicas_retenidas_filtro.png', 'notebook/figuras/18_radiomicas_retenidas_filtro.pdf']


[fig] 19_estabilidad_C_logistic  ->  ['notebook/figuras/19_estabilidad_C_logistic.png', 'notebook/figuras/19_estabilidad_C_logistic.pdf']
Figuras de la Seccion 6 generadas.


## 7 · Modelo final — caracterización (reproduce Etapa 5)

Discriminación, calibración, clasificación (umbral fijo 0.50, **no optimizado**) y DCA se
calculan sobre el **OOF agregado** de la Sección 6 (mismo criterio que Etapa 3/5). La
interpretabilidad (coeficientes/OR, loadings de PCA) reutiliza los ajustes ya obtenidos sobre
el **hold-out** en la Sección 5 (un único fit; exploratorio, no promediado entre folds).


In [9]:

EPS = 1e-6
FAMILY_RULES = [
    (r"^F\d+_", "F* (difusion/DT)"),
    (r"washin|washout|peak_enhancement|WashinRate|SER|PE_map|enhanc", "cinetica/realce"),
    (r"DFT|Moment_Invariant|Fourier", "Fourier/momentos"),
    (r"vol_cu_mm|Total_.*_vol|volume|Vol", "morfologia/volumen"),
    (r"Homogeneity|Entropy|Contrast|Correlation|Dissimilarity|inverse_difference|"
     r"information_measure|Sum_|Difference_|cluster|Energy|IDM|GLCM", "textura GLCM/1er orden"),
]

def family_of(name):
    for pat, lab in FAMILY_RULES:
        if re.search(pat, name, flags=re.I):
            return lab
    return "otras"

def logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))

def calibration_stats(y_, p):
    brier = brier_score_loss(y_, p)
    lp = logit(p)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp.reshape(-1, 1), y_)
    slope = float(lr.coef_[0][0])
    f = lambda a: np.mean(1 / (1 + np.exp(-(a + lp))) - y_)
    intercept = float(brentq(f, -10, 10))
    return {"brier": brier, "cal_slope": slope, "cal_intercept": intercept}

def clf_metrics(y_, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan
    return dict(threshold=thr, TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                sensitivity=sens, specificity=spec, PPV=ppv, NPV=npv, accuracy=(tp+tn)/len(y_))

def net_benefit(y_, p, pt_grid):
    n = len(y_); prev = y_.mean()
    nb_model, nb_all = [], []
    for pt in pt_grid:
        w = pt / (1 - pt)
        pred = (p >= pt).astype(int)
        tp = np.sum((pred == 1) & (y_ == 1)); fp = np.sum((pred == 1) & (y_ == 0))
        nb_model.append(tp / n - (fp / n) * w)
        nb_all.append(prev - (1 - prev) * w)
    return np.array(nb_model), np.array(nb_all)

yv = agg.y_true.values
pc, pcar = agg["C"].values, agg["C+A+R"].values

cal_c, cal_car = calibration_stats(yv, pc), calibration_stats(yv, pcar)
cls_c, cls_car = clf_metrics(yv, pc, 0.5), clf_metrics(yv, pcar, 0.5)
pt_grid = np.linspace(0.05, 0.6, 56)
nb_c, nb_all = net_benefit(yv, pc, pt_grid)
nb_car, _ = net_benefit(yv, pcar, pt_grid)

print("Calibracion (OOF agregado):"); display(pd.DataFrame([{"model": "C", **cal_c}, {"model": "C+A+R", **cal_car}]).set_index("model"))
print("\nClasificacion @0.50 (OOF agregado):"); display(pd.DataFrame([{"model": "C", **cls_c}, {"model": "C+A+R", **cls_car}]).set_index("model"))


Calibracion (OOF agregado):


,brier,cal_slope,cal_intercept
model,,,
C,0.202482,0.838004,-0.660119
C+A+R,0.197763,0.700704,-0.611793



Clasificacion @0.50 (OOF agregado):


,threshold,TN,FP,FN,TP,sensitivity,specificity,PPV,NPV,accuracy
model,,,,,,,,,,
C,0.5,124,64,30,63,0.677419,0.659574,0.496063,0.805195,0.665480
C+A+R,0.5,133,55,30,63,0.677419,0.707447,0.533898,0.815951,0.697509


In [10]:

# fig 20: calibration curve
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(5, 4.6))
for lab, p, col in [("C", pc, COLOR_C), ("C+A+R", pcar, COLOR_CAR)]:
    frac_pos, mean_pred = calibration_curve(yv, p, n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", color=col, label=lab)
ax.plot([0, 1], [0, 1], "--", color=COLOR_REF, lw=1)
ax.set_xlabel("probabilidad predicha"); ax.set_ylabel("frecuencia observada"); ax.legend()
save_figure(fig, "20_calibration_curve", "Curva de calibración, OOF agregado, C vs C+A+R",
            "Etapa 5", "SECUNDARIA", "Evalúa si la probabilidad predicha refleja el riesgo real")

# fig 22: DCA
fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.plot(pt_grid, nb_c, color=COLOR_C, lw=2, label="C")
ax.plot(pt_grid, nb_car, color=COLOR_CAR, lw=2, label="C+A+R")
ax.plot(pt_grid, nb_all, "--", color=COLOR_REF, lw=1.4, label="Treat all")
ax.axhline(0, color="black", lw=1, label="Treat none")
ax.set_xlabel("umbral de decisión (pt)"); ax.set_ylabel("beneficio neto"); ax.legend()
save_figure(fig, "22_dca_C_vs_CAR", "Decision Curve Analysis, C vs C+A+R, vs treat-all/treat-none",
            "Etapa 5", "SECUNDARIA", "Utilidad clínica potencial, no evidencia de superioridad")

# fig 23: matrices de confusion, misma escala
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
vmax = max(cls_c["TN"] + cls_c["FP"] + cls_c["FN"] + cls_c["TP"], 1)
for ax, lab, cls in zip(axes, ["C", "C+A+R"], [cls_c, cls_car]):
    mat = np.array([[cls["TN"], cls["FP"]], [cls["FN"], cls["TP"]]])
    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=max(cls_c["TP"]+cls_c["FN"], cls_car["TP"]+cls_car["FN"], cls_c["TN"]+cls_c["FP"]))
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(v), ha="center", va="center", color="black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["real 0", "real 1"])
    ax.set_title(lab)
fig.tight_layout()
save_figure(fig, "23_confusion_matrices_050", "Matrices de confusión a umbral 0.50, misma escala, C vs C+A+R",
            "Etapa 5", "SECUNDARIA", "Clasificación descriptiva a umbral fijo, no optimizado")

print("Umbral=0.50 (no optimizado). Metricas ya mostradas arriba.")


[fig] 20_calibration_curve  ->  ['notebook/figuras/20_calibration_curve.png', 'notebook/figuras/20_calibration_curve.pdf']


[fig] 22_dca_C_vs_CAR  ->  ['notebook/figuras/22_dca_C_vs_CAR.png', 'notebook/figuras/22_dca_C_vs_CAR.pdf']


[fig] 23_confusion_matrices_050  ->  ['notebook/figuras/23_confusion_matrices_050.png', 'notebook/figuras/23_confusion_matrices_050.pdf']
Umbral=0.50 (no optimizado). Metricas ya mostradas arriba.


In [11]:

# ---- Interpretabilidad: reutiliza el fit de hold-out (Seccion 5) ----
est_c, _ = holdout_fits["C"]
est_car, _ = holdout_fits["C+A+R"]

A_cols = set(sum((BLOCK_TYPES["A"].get(k, []) for k in ("continuous", "binary", "ordinal", "nominal")), []))

def feature_names_and_coef(est, blocks):
    pre = est.named_steps["preprocessor"]; clf = est.named_steps["clf"]
    names, blk = [], []
    for name, trans, cols in pre.transformers_:
        if name in ("R", "remainder") or isinstance(trans, str):
            continue
        if name == "nom":
            outc = trans.named_steps["ohe"].get_feature_names_out(cols).tolist()
        else:
            outc = list(cols)
        for c in outc:
            src = c.rsplit("_", 1)[0] if name == "nom" else c
            names.append(c); blk.append("A" if src in A_cols else "C")
    n_dense = len(names)
    coefs = clf.coef_[0]
    rows = [{"feature": names[i], "block": blk[i], "coef": float(coefs[i]), "odds_ratio": float(np.exp(coefs[i]))}
            for i in range(n_dense)]
    if "R" in blocks:
        for j in range(n_dense, len(coefs)):
            rows.append({"feature": f"PCA_R_{j - n_dense + 1}", "block": "R (PCA)",
                         "coef": float(coefs[j]), "odds_ratio": float(np.exp(coefs[j]))})
    return pd.DataFrame(rows)

def pca_loadings(est, rad_cols_, top=10):
    pre = est.named_steps["preprocessor"]
    Rp = pre.named_transformers_["R"]
    keep = Rp.named_steps["corr"].keep_
    kept_names = [rad_cols_[i] for i in keep]
    comps = Rp.named_steps["pca"].components_
    evr = Rp.named_steps["pca"].explained_variance_ratio_
    out = []
    for ci in range(comps.shape[0]):
        load = comps[ci]
        order = np.argsort(-np.abs(load))[:top]
        for rank, k in enumerate(order):
            out.append({"component": ci + 1, "rank": rank + 1, "radiomic": kept_names[k],
                        "loading": float(load[k]), "abs_loading": float(abs(load[k])), "family": family_of(kept_names[k])})
    return pd.DataFrame(out), evr, len(kept_names)

coef_c = feature_names_and_coef(est_c, ["C"])
coef_car = feature_names_and_coef(est_car, ["C", "A", "R"])
load_df, evr, n_kept = pca_loadings(est_car, rad_cols)

print("Coeficientes modelo C (top 10 por |coef|):")
display(coef_c.reindex(coef_c.coef.abs().sort_values(ascending=False).index).head(10))

# fig 24: coeficientes/OR de C
top_c = coef_c.reindex(coef_c.coef.abs().sort_values(ascending=False).index).head(12)
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top_c.feature[::-1], top_c.coef[::-1], color=COLOR_C)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("coeficiente (logit)")
save_figure(fig, "24_coeficientes_modelo_C", "Coeficientes del modelo clínico C (top 12 por |coef|)",
            "Etapa 5", "SUPLEMENTARIA", "Interpretabilidad del modelo clínico de referencia")

# fig 25: contribucion por bloque en C+A+R (descriptivo, sin causalidad)
fig, ax = plt.subplots(figsize=(6, 4))
block_contrib = coef_car.assign(abs_coef=coef_car.coef.abs()).groupby("block")["abs_coef"].sum()
colors_blk = {"C": COLOR_C, "A": COLOR_M, "R (PCA)": COLOR_CAR}
ax.bar(block_contrib.index, block_contrib.values, color=[colors_blk.get(b, COLOR_REF) for b in block_contrib.index])
ax.set_ylabel("suma de |coeficiente| (descriptivo)")
save_figure(fig, "25_contribucion_por_bloque_CAR", "Suma de |coeficiente| por bloque en C+A+R (descriptivo, no causal)",
            "Etapa 5", "SUPLEMENTARIA", "Ilustra pesos relativos; no permite atribución causal")

# fig 26: varianza explicada PCA
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.bar(np.arange(1, len(evr) + 1), evr, color=COLOR_CAR)
ax.set_xlabel("componente PCA"); ax.set_ylabel("varianza explicada")
save_figure(fig, "26_pca_varianza_explicada", f"Varianza explicada por componente PCA (C+A+R, hold-out, {n_kept} radiómicas tras filtro)",
            "Etapa 5", "SUPLEMENTARIA", "Contexto de cuánta información captura cada PC")

# fig 27: familias dominantes en loadings
fig, ax = plt.subplots(figsize=(6.5, 4.2))
fam_counts = load_df[load_df["rank"] <= 5].groupby("family").size().sort_values(ascending=False)
ax.barh(fam_counts.index[::-1], fam_counts.values[::-1], color=COLOR_CAR)
ax.set_xlabel("apariciones en top-5 loadings de cada PC")
save_figure(fig, "27_familias_dominantes_pca", "Familias radiómicas dominantes en los loadings de PCA (top-5 por componente)",
            "Etapa 5", "SUPLEMENTARIA", "Describe qué tipo de radiómicas pesan más en los PCs, sin implicar biomarcador individual")

print("Interpretabilidad (Seccion 7) completa.")


Coeficientes modelo C (top 10 por |coef|):


,feature,block,coef,odds_ratio
3,HER2_bin,C,0.758215,2.134464
2,PR_bin,C,-0.535085,0.585619
1,ER_bin,C,-0.393124,0.674945
5,tumor_side_R,C,-0.343301,0.709425
10,grade_mitotic,C,0.252469,1.287200
12,race_2.0,C,-0.245485,0.782325
9,grade_nuclear,C,0.218701,1.244459
8,grade_tubule,C,0.173347,1.189279
0,age,C,-0.162927,0.849653
16,race_6.0,C,0.156710,1.169657


[fig] 24_coeficientes_modelo_C  ->  ['notebook/figuras/24_coeficientes_modelo_C.png', 'notebook/figuras/24_coeficientes_modelo_C.pdf']


[fig] 25_contribucion_por_bloque_CAR  ->  ['notebook/figuras/25_contribucion_por_bloque_CAR.png', 'notebook/figuras/25_contribucion_por_bloque_CAR.pdf']


[fig] 26_pca_varianza_explicada  ->  ['notebook/figuras/26_pca_varianza_explicada.png', 'notebook/figuras/26_pca_varianza_explicada.pdf']


[fig] 27_familias_dominantes_pca  ->  ['notebook/figuras/27_familias_dominantes_pca.png', 'notebook/figuras/27_familias_dominantes_pca.pdf']
Interpretabilidad (Seccion 7) completa.


## 8 · Algoritmos — Random Forest y Gradient Boosting (comparación exploratoria)

Reproduce Etapa 4 (versión "liviana", tal como quedó congelada tras la interrupción de la
versión pesada —. **Esquema deliberadamente más liviano que el de Logistic**:
Nested CV **única** (no repetida), outer `StratifiedKFold(5)` × inner `StratifiedKFold(3)`,
`random_state=42`. Grillas preespecificadas, sobre C y C+A+R únicamente.


In [12]:

ALGOS = {
    "random_forest": dict(
        make=lambda: RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=1),
        grid={"clf__max_depth": [None, 3, 5], "clf__min_samples_leaf": [2, 5, 10],
              "clf__max_features": ["sqrt", 0.5], "clf__class_weight": [None, "balanced"]},
        scale_dense=False),
    "gradient_boosting": dict(
        make=lambda: GradientBoostingClassifier(random_state=RANDOM_STATE),
        grid={"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1],
              "clf__max_depth": [2, 3], "clf__min_samples_leaf": [10]},
        scale_dense=False),
}
MODELS_S4 = {"C": ["C"], "C+A+R": ["C", "A", "R"]}

def make_preprocessor4(blocks, scale_dense):
    cont, binc, ordc, nomc = [], [], [], []
    for b in ("C", "A"):
        if b in blocks:
            t = BLOCK_TYPES[b]
            cont += t.get("continuous", []); binc += t.get("binary", [])
            ordc += t.get("ordinal", []); nomc += t.get("nominal", [])
    tr = []
    if cont:
        steps = [("imp", SimpleImputer(strategy="median"))]
        if scale_dense: steps.append(("sc", StandardScaler()))
        tr.append(("cont", Pipeline(steps), cont))
    if binc:
        tr.append(("bin", SimpleImputer(strategy="most_frequent"), binc))
    if ordc:
        steps = [("imp", SimpleImputer(strategy="median"))]
        if scale_dense: steps.append(("sc", StandardScaler()))
        tr.append(("ord", Pipeline(steps), ordc))
    if nomc:
        tr.append(("nom", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), nomc))
    if "R" in blocks:
        tr.append(("R", Pipeline([("imp", SimpleImputer(strategy="median")),
                                  ("corr", CorrelationFilter(CORR_THRESHOLD)), ("sc", StandardScaler()),
                                  ("pca", PCA(random_state=RANDOM_STATE))]), rad_cols))
    return ColumnTransformer(tr, remainder="drop")

def make_estimator4(algo, blocks):
    spec = ALGOS[algo]
    return Pipeline([("preprocessor", make_preprocessor4(blocks, spec["scale_dense"])), ("clf", spec["make"]())])

def make_grid4(algo, blocks):
    grid = dict(ALGOS[algo]["grid"])
    if "R" in blocks:
        grid["preprocessor__R__pca__n_components"] = NCOMP_GRID
    return grid

algo_rows = []
algo_oof = {}
N_OUTER4, N_INNER4 = 5, 3
for algo in ALGOS:
    for m, blocks in MODELS_S4.items():
        outer = StratifiedKFold(n_splits=N_OUTER4, shuffle=True, random_state=RANDOM_STATE)
        oof_p = np.full(len(merged), np.nan)
        for tri, tei in outer.split(merged, y.values):
            gs = GridSearchCV(make_estimator4(algo, blocks), make_grid4(algo, blocks), scoring="roc_auc",
                              cv=StratifiedKFold(n_splits=N_INNER4, shuffle=True, random_state=RANDOM_STATE),
                              n_jobs=-1, refit=True)
            gs.fit(merged.iloc[tri], y.iloc[tri])
            oof_p[tei] = gs.predict_proba(merged.iloc[tei])[:, 1]
        algo_oof[(algo, m)] = oof_p
        algo_rows.append({"algo": algo, "model": m, "roc_auc": roc_auc_score(y.values, oof_p),
                          "pr_auc": average_precision_score(y.values, oof_p)})
        print(f"  [{algo:17s}] {m:6s} ROC={algo_rows[-1]['roc_auc']:.3f}  PR={algo_rows[-1]['pr_auc']:.3f}")

algo_df = pd.DataFrame(algo_rows)
# fila de referencia: Logistic, bajo SU esquema propio (Repeated Nested CV, OOF agregado) -- no recalculado, ya en agg_m
algo_df = pd.concat([algo_df, pd.DataFrame([
    {"algo": "logistic", "model": "C", "roc_auc": agg_m["C"]["roc_auc_agg"], "pr_auc": agg_m["C"]["pr_auc_agg"]},
    {"algo": "logistic", "model": "C+A+R", "roc_auc": agg_m["C+A+R"]["roc_auc_agg"], "pr_auc": agg_m["C+A+R"]["pr_auc_agg"]},
])], ignore_index=True)
display(algo_df.pivot(index="algo", columns="model", values=["roc_auc", "pr_auc"]))
print("\nNOTA: Logistic bajo Repeated Nested CV (10 repeticiones); RF/GB bajo Nested CV UNICA "
      "(outer5xinner3, seed 42) -- esquemas de validacion distintos, NO comparables como si fueran identicos.")


  [random_forest    ] C      ROC=0.726  PR=0.522


  [random_forest    ] C+A+R  ROC=0.701  PR=0.543


  [gradient_boosting] C      ROC=0.714  PR=0.512


  [gradient_boosting] C+A+R  ROC=0.748  PR=0.558


roc_auc              pr_auc          
model                     C     C+A+R         C     C+A+R
algo                                                     
gradient_boosting  0.713738  0.747541  0.511970  0.557654
logistic           0.749886  0.761039  0.546409  0.576586
random_forest      0.726493  0.700583  0.521980  0.543272


NOTA: Logistic bajo Repeated Nested CV (10 repeticiones); RF/GB bajo Nested CV UNICA (outer5xinner3, seed 42) -- esquemas de validacion distintos, NO comparables como si fueran identicos.


In [13]:

# fig 28/29: ROC y PR por algoritmo, C vs C+A+R
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, algo, col in zip(axes, ["logistic", "random_forest", "gradient_boosting"], [COLOR_CAR, COLOR_RF, COLOR_GB]):
    sub = algo_df[algo_df.algo == algo].set_index("model")
    ax.bar(["C", "C+A+R"], [sub.loc["C", "roc_auc"], sub.loc["C+A+R", "roc_auc"]], color=[COLOR_C, col])
    ax.set_title(algo); ax.set_ylim(0.5, 0.85)
axes[0].set_ylabel("ROC-AUC")
fig.suptitle("ROC-AUC por algoritmo (Logistic: validación anidada repetida 10×5×5; RF/GB: anidada única 5×3)", fontsize=10)
save_figure(fig, "28_roc_por_algoritmo", "ROC-AUC C vs C+A+R por algoritmo (esquemas de validación distintos)",
            "Etapa 4 (light)", "SECUNDARIA", "Compara algoritmos, marcando validación más liviana en RF/GB")

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, algo, col in zip(axes, ["logistic", "random_forest", "gradient_boosting"], [COLOR_CAR, COLOR_RF, COLOR_GB]):
    sub = algo_df[algo_df.algo == algo].set_index("model")
    ax.bar(["C", "C+A+R"], [sub.loc["C", "pr_auc"], sub.loc["C+A+R", "pr_auc"]], color=[COLOR_C, col])
    ax.set_title(algo); ax.set_ylim(0.3, 0.7)
axes[0].set_ylabel("PR-AUC")
fig.suptitle("PR-AUC por algoritmo (Logistic: validación anidada repetida 10×5×5; RF/GB: anidada única 5×3)", fontsize=10)
save_figure(fig, "29_pr_por_algoritmo", "PR-AUC C vs C+A+R por algoritmo (esquemas de validación distintos)",
            "Etapa 4 (light)", "SECUNDARIA", "Compara algoritmos, marcando validación más liviana en RF/GB")

# fig 30: delta A+R por algoritmo
fig, ax = plt.subplots(figsize=(6, 4))
deltas_roc, deltas_pr, algos_ = [], [], []
for algo in ["logistic", "random_forest", "gradient_boosting"]:
    sub = algo_df[algo_df.algo == algo].set_index("model")
    deltas_roc.append(sub.loc["C+A+R", "roc_auc"] - sub.loc["C", "roc_auc"])
    deltas_pr.append(sub.loc["C+A+R", "pr_auc"] - sub.loc["C", "pr_auc"])
    algos_.append(algo)
xw = np.arange(3)
ax.bar(xw - 0.18, deltas_roc, width=0.35, label="ΔROC-AUC", color=COLOR_CAR)
ax.bar(xw + 0.18, deltas_pr, width=0.35, label="ΔPR-AUC", color=COLOR_CR)
ax.axhline(0, color="black", lw=1)
ax.set_xticks(xw); ax.set_xticklabels(algos_); ax.legend()
save_figure(fig, "30_delta_AR_por_algoritmo", "Δ (C+A+R − C) por algoritmo, ROC-AUC y PR-AUC",
            "Etapa 4 (light)", "SECUNDARIA", "Evidencia de que el incremento no es exclusivo del modelo lineal")

print("Figuras de la Seccion 8 generadas.")


[fig] 28_roc_por_algoritmo  ->  ['notebook/figuras/28_roc_por_algoritmo.png', 'notebook/figuras/28_roc_por_algoritmo.pdf']


[fig] 29_pr_por_algoritmo  ->  ['notebook/figuras/29_pr_por_algoritmo.png', 'notebook/figuras/29_pr_por_algoritmo.pdf']
[fig] 30_delta_AR_por_algoritmo  ->  ['notebook/figuras/30_delta_AR_por_algoritmo.png', 'notebook/figuras/30_delta_AR_por_algoritmo.pdf']
Figuras de la Seccion 8 generadas.


## 9 · Resumen principal

Tablas finales, generadas directamente de las variables ya calculadas en este notebook
(no hardcodeadas).


In [14]:

resumen_principal = pd.DataFrame({
    "modelo": ["C", "C+R", "C+A+R"],
    "holdout_ROC": [holdout_df.loc[m, "roc_auc"] for m in ["C", "C+R", "C+A+R"]],
    "holdout_PR": [holdout_df.loc[m, "pr_auc"] for m in ["C", "C+R", "C+A+R"]],
    "repCV_ROC_mean": [summ.loc[m, "roc_auc_mean"] for m in ["C", "C+R", "C+A+R"]],
    "repCV_ROC_sd": [summ.loc[m, "roc_auc_sd"] for m in ["C", "C+R", "C+A+R"]],
    "repCV_PR_mean": [summ.loc[m, "pr_auc_mean"] for m in ["C", "C+R", "C+A+R"]],
    "repCV_PR_sd": [summ.loc[m, "pr_auc_sd"] for m in ["C", "C+R", "C+A+R"]],
    "OOF_ROC_agg": [agg_m[m]["roc_auc_agg"] for m in ["C", "C+R", "C+A+R"]],
    "OOF_PR_agg": [agg_m[m]["pr_auc_agg"] for m in ["C", "C+R", "C+A+R"]],
}).set_index("modelo")
print("Resumen principal (Stricter, N=281) -- NO mezclar columnas entre si:")
display(resumen_principal)
print("\nBootstrap pareado (Delta vs C, OOF agregado):")
display(boot)


Resumen principal (Stricter, N=281) -- NO mezclar columnas entre si:


,holdout_ROC,holdout_PR,repCV_ROC_mean,repCV_ROC_sd,repCV_PR_mean,repCV_PR_sd,OOF_ROC_agg,OOF_PR_agg
modelo,,,,,,,,
C,0.778393,0.633772,0.741261,0.011612,0.541588,0.016613,0.749886,0.546409
C+R,0.745152,0.555596,0.739333,0.016264,0.536933,0.025192,0.748856,0.549439
C+A+R,0.783934,0.708269,0.742896,0.018570,0.555483,0.027062,0.761039,0.576586



Bootstrap pareado (Delta vs C, OOF agregado):


,comparison,delta_roc_point,delta_roc_ci_lo,delta_roc_ci_hi,delta_pr_point,delta_pr_ci_lo,delta_pr_ci_hi,n_boot
0,C vs C+R,-0.001030,-0.012719,0.010902,0.003030,-0.018216,0.023361,5000
1,C vs C+A+R,0.011153,-0.024411,0.048385,0.030176,-0.032654,0.093796,5000


## 10 · Conclusiones computacionales

Resumen estrictamente derivado de los resultados calculados en este notebook (ver Sección 9
para las tablas completas).


In [15]:

d_roc_car = agg_m["C+A+R"]["roc_auc_agg"] - agg_m["C"]["roc_auc_agg"]
d_pr_car = agg_m["C+A+R"]["pr_auc_agg"] - agg_m["C"]["pr_auc_agg"]
boot_car = boot[boot.comparison == "C vs C+A+R"].iloc[0]

print(f"""
- Cohorte final: N={len(merged)} (188 no respondedores / 93 respondedores, prevalencia 33.1%).
- Modelo clinico C (Repeated Nested CV, OOF agregado): ROC-AUC={agg_m['C']['roc_auc_agg']:.3f},
  PR-AUC={agg_m['C']['pr_auc_agg']:.3f}.
- Modelo C+A+R (OOF agregado): ROC-AUC={agg_m['C+A+R']['roc_auc_agg']:.3f},
  PR-AUC={agg_m['C+A+R']['pr_auc_agg']:.3f}.
- Delta (C+A+R - C), OOF agregado: ROC-AUC {d_roc_car:+.3f}, PR-AUC {d_pr_car:+.3f}.
- Bootstrap pareado (5000 resamples): ROC-AUC IC95% [{boot_car.delta_roc_ci_lo:+.3f}, {boot_car.delta_roc_ci_hi:+.3f}]
  ({"incluye" if boot_car.delta_roc_ci_lo <= 0 <= boot_car.delta_roc_ci_hi else "NO incluye"} 0);
  PR-AUC IC95% [{boot_car.delta_pr_ci_lo:+.3f}, {boot_car.delta_pr_ci_hi:+.3f}]
  ({"incluye" if boot_car.delta_pr_ci_lo <= 0 <= boot_car.delta_pr_ci_hi else "NO incluye"} 0).
- Lectura: si los IC95% incluyen 0, la diferencia observada no es concluyente y no se afirma
  superioridad de C+A+R sobre C. A se incorpora como contexto tecnico de adquisicion de R; no se
  atribuye causalmente a A ninguna diferencia.
- RF y GB se evaluaron con una validacion mas liviana (Seccion 8): comparacion exploratoria,
  no directamente equiparable a la Repeated Nested CV de la regresion logistica.
""")



- Cohorte final: N=281 (188 no respondedores / 93 respondedores, prevalencia 33.1%).
- Modelo clinico C (Repeated Nested CV, OOF agregado): ROC-AUC=0.750,
  PR-AUC=0.546.
- Modelo C+A+R (OOF agregado): ROC-AUC=0.761,
  PR-AUC=0.577.
- Delta (C+A+R - C), OOF agregado: ROC-AUC +0.011, PR-AUC +0.030.
- Bootstrap pareado (5000 resamples): ROC-AUC IC95% [-0.024, +0.048]
  (incluye 0);
  PR-AUC IC95% [-0.033, +0.094]
  (incluye 0).
- Lectura: si los IC95% incluyen 0, la diferencia observada no es concluyente y no se afirma
  superioridad de C+A+R sobre C. A se incorpora como contexto tecnico de adquisicion de R; no se
  atribuye causalmente a A ninguna diferencia.
- RF y GB se evaluaron con una validacion mas liviana (Seccion 8): comparacion exploratoria,
  no directamente equiparable a la Repeated Nested CV de la regresion logistica.



## 11 · Catálogo de figuras

In [16]:

catalog_df = pd.DataFrame(FIGURE_CATALOG)
catalog_df.to_csv(FIG_DIR / "catalogo_figuras.csv", index=False)
print(f"Total de figuras generadas: {len(catalog_df)}")
display(catalog_df[["figura", "archivo", "que_muestra", "analisis", "prioridad", "uso_potencial"]])


Total de figuras generadas: 23


,figura,archivo,que_muestra,analisis,prioridad,uso_potencial
0,06_roc_oof_C_vs_CAR,notebook/figuras/06_roc_oof_C_vs_CAR.png,"Curva ROC sobre OOF agregado, C vs C+A+R (Stri...",Etapa 3 (Repeated Nested CV),PRINCIPAL,Figura principal de desempeño del TFG
1,07_pr_oof_C_vs_CAR,notebook/figuras/07_pr_oof_C_vs_CAR.png,"Curva PR sobre OOF agregado, C vs C+A+R + base...",Etapa 3 (Repeated Nested CV),PRINCIPAL,Figura principal de desempeño del TFG
2,08_roc_C_CR_CAR,notebook/figuras/08_roc_C_CR_CAR.png,"Curva ROC OOF agregado: C, C+R, C+A+R",Etapa 3,PRINCIPAL,Muestra la incrementalidad R y A+R
3,09_pr_C_CR_CAR,notebook/figuras/09_pr_C_CR_CAR.png,"Curva PR OOF agregado: C, C+R, C+A+R",Etapa 3,PRINCIPAL,Muestra la incrementalidad R y A+R
4,10_holdout_vs_robusto,notebook/figuras/10_holdout_vs_robusto.png,ROC-AUC: hold-out histórico vs OOF agregado (R...,Etapas 2 y 3,PRINCIPAL,"Aclara que el hold-out es solo comparabilidad,..."
5,11_delta_bootstrap_forest,notebook/figuras/11_delta_bootstrap_forest.png,Δ ROC/PR-AUC (C+R y C+A+R vs C) con IC95% boot...,Etapa 3,PRINCIPAL,Resume incertidumbre de la incrementalidad
6,12_roc_por_repeticion,notebook/figuras/12_roc_por_repeticion.png,ROC-AUC por repetición (10) para C/C+R/C+A+R,Etapa 3,SECUNDARIA,Muestra dispersión entre repeticiones
7,13_pr_por_repeticion,notebook/figuras/13_pr_por_repeticion.png,PR-AUC por repetición (10) para C/C+R/C+A+R,Etapa 3,SECUNDARIA,Muestra dispersión entre repeticiones
8,14_delta_roc_por_repeticion,notebook/figuras/14_delta_roc_por_repeticion.png,"Δ ROC-AUC vs C, por repetición",Etapa 3,SECUNDARIA,Estabilidad de la incrementalidad entre repeti...
9,15_delta_pr_por_repeticion,notebook/figuras/15_delta_pr_por_repeticion.png,"Δ PR-AUC vs C, por repetición",Etapa 3,SECUNDARIA,Estabilidad de la incrementalidad entre repeti...


## Validación final (checklist)

1. Este notebook parte únicamente de `datos/Clinical_and_Other_Features.xlsx` y
   `datos/Imaging_Features.xlsx` (Sección 1) — ningún CSV de `pipeline_v2/` se usó para calcular
   resultados, sólo para comparar al final (ver bloque siguiente).
2. N/clases verificados por `assert` en la Sección 2 (N=281, 93/188).
3. Todo el preprocessing dependiente de train (imputación, correlation filter, PCA, tuning de
   `C`) está dentro de `Pipeline`/`ColumnTransformer` ajustado por fold — sin leakage.
4. Seeds: `RANDOM_STATE=42` (hold-out y grids), `REP_SEEDS=42..51` (Repeated Nested CV, idénticos
   a Etapas 3/4), `BOOT_SEED=20260909` (bootstrap) — ninguno fue elegido ahora, todos
   reutilizados de las etapas congeladas.
5. Paths relativos a la raíz del proyecto (`Path.cwd()`), sin rutas absolutas hardcodeadas.


In [17]:

# ---- comparacion contra los CSV congelados (SOLO validacion, no sustituye el calculo) ----
try:
    ref_summ = pd.read_csv(PIPE_DIR / "stage3_summary.csv").set_index("model")
    ref_agg = pd.read_csv(PIPE_DIR / "stage3_oof_aggregated.csv")
    ref_roc_c = roc_auc_score(ref_agg.y_true, ref_agg["C"])
    ref_roc_car = roc_auc_score(ref_agg.y_true, ref_agg["C+A+R"])
    print("Comparacion OOF agregado ROC-AUC (notebook vs congelado Etapa 3):")
    print(f"  C:      notebook={agg_m['C']['roc_auc_agg']:.4f}   congelado={ref_roc_c:.4f}   "
          f"diff={agg_m['C']['roc_auc_agg']-ref_roc_c:+.4f}")
    print(f"  C+A+R:  notebook={agg_m['C+A+R']['roc_auc_agg']:.4f}   congelado={ref_roc_car:.4f}   "
          f"diff={agg_m['C+A+R']['roc_auc_agg']-ref_roc_car:+.4f}")
    print("\nSi la diferencia no es ~0, DOCUMENTAR la causa (no corregir silenciosamente).")
except FileNotFoundError as e:
    print("Archivo congelado no encontrado para comparar:", e)


Comparacion OOF agregado ROC-AUC (notebook vs congelado Etapa 3):
  C:      notebook=0.7499   congelado=0.7499   diff=+0.0000
  C+A+R:  notebook=0.7610   congelado=0.7610   diff=+0.0000

Si la diferencia no es ~0, DOCUMENTAR la causa (no corregir silenciosamente).
